# ЛР 1.2 — Сплайн-интерполяция табличных функций

## Вариант 13
- **Тип сплайна:** с отсутствием узла (аналог `not-a-knot`)
- **f(x):** $(x-0.5)^3\ln(x)$, $x\in[0.3,\,0.8]$
- **g(x):** $\exp(\cos(3x))$, $x\in[0,\,\pi]$

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 11
plt.rcParams["figure.figsize"] = (10, 5)
np.set_printoptions(precision=6, suppress=True)

## функции

- $f(x)=(x-0.5)^3\ln(x)$ на $[0.3, 0.8]$ — для исследования сходимости.
- $g(x)=\exp(\cos(3x))$ на $[0,\pi]$ — для сравнения полинома и сплайна.

In [ ]:
def f(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    return (x - 0.5) ** 3 * np.log(x)


def g(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    return np.exp(np.cos(3.0 * x))


A_F, B_F = 0.3, 0.8
A_G, B_G = 0.0, np.pi

## Реализация

В этом пункте реализованы:
- интерполяционный полином Ньютона,
- кубический сплайн (включая `not-a-knot`),
- вычисление значения полинома и сплайна в заданной точке,
- вычисление RMSE.

In [ ]:
def uniform_nodes(a: float, b: float, n: int) -> np.ndarray:
    return np.linspace(a, b, n)


def chebyshev_nodes(a: float, b: float, n: int) -> np.ndarray:
    k = np.arange(1, n + 1)
    t = np.cos((2.0 * k - 1.0) / (2.0 * n) * np.pi)
    return 0.5 * (a + b) + 0.5 * (b - a) * t


def newton_divided_differences(x_nodes: np.ndarray, y_nodes: np.ndarray) -> np.ndarray:
    x_nodes = np.asarray(x_nodes, dtype=float)
    coef = np.asarray(y_nodes, dtype=float).copy()
    n = len(x_nodes)
    for j in range(1, n):
        coef[j:n] = (coef[j:n] - coef[j - 1 : n - 1]) / (x_nodes[j:n] - x_nodes[0 : n - j])
    return coef


def newton_eval(x_nodes: np.ndarray, coef: np.ndarray, x_eval: np.ndarray) -> np.ndarray:
    x_nodes = np.asarray(x_nodes, dtype=float)
    coef = np.asarray(coef, dtype=float)
    x_eval = np.asarray(x_eval, dtype=float)
    y = np.full_like(x_eval, coef[-1], dtype=float)
    for k in range(len(coef) - 2, -1, -1):
        y = y * (x_eval - x_nodes[k]) + coef[k]
    return y


@dataclass
class CubicSplineCoefficients:
    x_nodes: np.ndarray
    a: np.ndarray
    b: np.ndarray
    c: np.ndarray
    d: np.ndarray


def build_cubic_spline(
    x_nodes: np.ndarray,
    y_nodes: np.ndarray,
    bc_type: str = "not-a-knot",
    fp0: float | None = None,
    fpn: float | None = None,
) -> CubicSplineCoefficients:
    x = np.asarray(x_nodes, dtype=float)
    y = np.asarray(y_nodes, dtype=float)
    n = len(x)

    if n < 2:
        raise ValueError("Нужно минимум 2 узла")
    if np.any(np.diff(x) <= 0):
        raise ValueError("x_nodes должны быть строго возрастающими")

    h = np.diff(x)
    A = np.zeros((n, n), dtype=float)
    rhs = np.zeros(n, dtype=float)

    for i in range(1, n - 1):
        A[i, i - 1] = h[i - 1]
        A[i, i] = 2.0 * (h[i - 1] + h[i])
        A[i, i + 1] = h[i]
        rhs[i] = 6.0 * ((y[i + 1] - y[i]) / h[i] - (y[i] - y[i - 1]) / h[i - 1])

    if bc_type == "natural":
        A[0, 0] = 1.0
        A[-1, -1] = 1.0
    elif bc_type == "clamped":
        if fp0 is None or fpn is None:
            raise ValueError("Для clamped нужно задать fp0 и fpn")
        A[0, 0] = 2.0 * h[0]
        A[0, 1] = h[0]
        rhs[0] = 6.0 * ((y[1] - y[0]) / h[0] - fp0)

        A[-1, -2] = h[-1]
        A[-1, -1] = 2.0 * h[-1]
        rhs[-1] = 6.0 * (fpn - (y[-1] - y[-2]) / h[-1])
    elif bc_type == "not-a-knot":
        if n < 4:
            A[0, 0] = 1.0
            A[-1, -1] = 1.0
        else:
            A[0, 0] = -h[1]
            A[0, 1] = h[0] + h[1]
            A[0, 2] = -h[0]
            rhs[0] = 0.0

            A[-1, -3] = -h[-1]
            A[-1, -2] = h[-2] + h[-1]
            A[-1, -1] = -h[-2]
            rhs[-1] = 0.0
    else:
        raise ValueError("Неизвестный bc_type")

    m = np.linalg.solve(A, rhs)

    a = y[:-1].copy()
    b = np.zeros(n - 1, dtype=float)
    c = np.zeros(n - 1, dtype=float)
    d = np.zeros(n - 1, dtype=float)

    for i in range(n - 1):
        b[i] = (y[i + 1] - y[i]) / h[i] - h[i] * (2.0 * m[i] + m[i + 1]) / 6.0
        c[i] = m[i] / 2.0
        d[i] = (m[i + 1] - m[i]) / (6.0 * h[i])

    return CubicSplineCoefficients(x_nodes=x, a=a, b=b, c=c, d=d)


def spline_eval(spl: CubicSplineCoefficients, x_eval: np.ndarray) -> np.ndarray:
    x_eval = np.asarray(x_eval, dtype=float)
    x = spl.x_nodes
    n = len(x)

    idx = np.searchsorted(x, x_eval, side="right") - 1
    idx = np.clip(idx, 0, n - 2)
    dx = x_eval - x[idx]
    return spl.a[idx] + spl.b[idx] * dx + spl.c[idx] * dx**2 + spl.d[idx] * dx**3


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

In [ ]:
# Демонстрация коэффициентов сплайна и вычисления в точке
n_demo = 10
x_demo = uniform_nodes(A_F, B_F, n_demo)
y_demo = f(x_demo)

coef_demo = newton_divided_differences(x_demo, y_demo)
spl_demo = build_cubic_spline(x_demo, y_demo, bc_type="not-a-knot")

coeff_df = pd.DataFrame({
    "i": np.arange(n_demo - 1),
    "a_i": spl_demo.a,
    "b_i": spl_demo.b,
    "c_i": spl_demo.c,
    "d_i": spl_demo.d,
})
print("Коэффициенты кубического сплайна (f, n=10):")
display(coeff_df)

x_test = np.array([0.34, 0.51, 0.77])
check_df = pd.DataFrame({
    "x": x_test,
    "f(x)": f(x_test),
    "P_newton(x)": newton_eval(x_demo, coef_demo, x_test),
    "S_spline(x)": spline_eval(spl_demo, x_test),
})
print("Проверка вычисления значения в точке:")
display(check_df)

## Эксперимент: сравнение полинома и сплайна на функции $g(x)$

- Узлы: равномерные, $n \in \{5, 10, 15, 20\}$.
- На каждом $n$ строим:
  - интерполяционный полином Ньютона,
  - кубический сплайн (`not-a-knot`).
- Считаем RMSE и визуализируем кривые на одном графике.

In [ ]:
n_values_2 = [5, 10, 15, 20]
x_dense_g = np.linspace(A_G, B_G, 2000)
y_true_g = g(x_dense_g)

rmse_rows = []
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, n in enumerate(n_values_2):
    x_nodes = uniform_nodes(A_G, B_G, n)
    y_nodes = g(x_nodes)

    coef = newton_divided_differences(x_nodes, y_nodes)
    y_poly = newton_eval(x_nodes, coef, x_dense_g)

    spl = build_cubic_spline(x_nodes, y_nodes, bc_type="not-a-knot")
    y_spline = spline_eval(spl, x_dense_g)

    e_poly = rmse(y_true_g, y_poly)
    e_spline = rmse(y_true_g, y_spline)
    rmse_rows.append({"n": n, "RMSE полином": e_poly, "RMSE сплайн": e_spline})

    ax = axes[i]
    ax.plot(x_dense_g, y_true_g, "k", lw=2, label="g(x)")
    ax.plot(x_dense_g, y_poly, "--", lw=1.5, label="Полином Ньютона")
    ax.plot(x_dense_g, y_spline, "-.", lw=1.5, label="Кубический сплайн")
    ax.scatter(x_nodes, y_nodes, c="red", s=20, label="Узлы")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("Аргумент x")
    ax.set_ylabel("Значение функции")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend()

plt.suptitle("Сравнение полинома и сплайна для g(x) = exp(cos(3x))", fontsize=14)
plt.tight_layout()
plt.show()

rmse_task2_df = pd.DataFrame(rmse_rows)
print("RMSE для пункта 2:")
display(rmse_task2_df)

## Скорость сходимости сплайнов (равномерные узлы)

Для функции $f(x)$ строим сплайн на последовательности $n = 5, 10, 20, 40, 80$.

- Считаем RMSE на плотной сетке.
- Строим график `log(RMSE)` от `log(n)`.
- Экспериментальная скорость сходимости: наклон прямой в log-log масштабе.

In [ ]:
def spline_convergence(func, a: float, b: float, n_values: list[int], node_kind: str):
    x_dense = np.linspace(a, b, 3000)
    y_true = func(x_dense)
    errors = []

    for n in n_values:
        if node_kind == "uniform":
            x_nodes = uniform_nodes(a, b, n)
        elif node_kind == "chebyshev":
            x_nodes = chebyshev_nodes(a, b, n)
            x_nodes.sort()
        else:
            raise ValueError("node_kind должен быть uniform или chebyshev")

        y_nodes = func(x_nodes)
        spl = build_cubic_spline(x_nodes, y_nodes, bc_type="not-a-knot")
        y_hat = spline_eval(spl, x_dense)
        errors.append(rmse(y_true, y_hat))

    n_arr = np.asarray(n_values, dtype=float)
    err_arr = np.asarray(errors, dtype=float)
    slope, intercept = np.polyfit(np.log(n_arr), np.log(err_arr), 1)
    return n_arr, err_arr, float(slope), float(intercept)


n_values_34 = [5, 10, 20, 40, 80]
n_u, err_u, slope_u, b_u = spline_convergence(f, A_F, B_F, n_values_34, node_kind="uniform")

uniform_df = pd.DataFrame({"n": n_u.astype(int), "RMSE (равномерные)": err_u})
print(f"Наклон для равномерных узлов: {slope_u:.6f}")
display(uniform_df)

plt.figure(figsize=(8, 5))
plt.loglog(n_u, err_u, "o-", lw=1.8, label=f"Равномерные узлы, наклон={slope_u:.3f}")
plt.xlabel("Число узлов n (лог. шкала)")
plt.ylabel("RMSE (лог. шкала)")
plt.title("Сходимость сплайна для f(x): равномерные узлы")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

## Выбор узлов: повтор для узлов Чебышёва

Повторяем исследование сходимости сплайна для $f(x)$, заменив равномерные узлы на узлы Чебышёва, затем сравниваем результаты.

In [ ]:
n_c, err_c, slope_c, b_c = spline_convergence(f, A_F, B_F, n_values_34, node_kind="chebyshev")

compare_df = pd.DataFrame({
    "n": n_u.astype(int),
    "RMSE (равномерные)": err_u,
    "RMSE (Чебышёв)": err_c,
})
print(f"Наклон (равномерные): {slope_u:.6f}")
print(f"Наклон (Чебышёв):    {slope_c:.6f}")
display(compare_df)

plt.figure(figsize=(9, 6))
plt.loglog(n_u, err_u, "o-", lw=1.8, label=f"Равномерные узлы, наклон={slope_u:.3f}")
plt.loglog(n_c, err_c, "s-", lw=1.8, label=f"Узлы Чебышёва, наклон={slope_c:.3f}")
plt.xlabel("Число узлов n (лог. шкала)")
plt.ylabel("RMSE (лог. шкала)")
plt.title("Сравнение сходимости сплайна для f(x)")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

## Анализ результатов и выводы

После запуска всех ячеек выше можно оформить выводы так:

1. **Полином Ньютона vs сплайн на $g(x)$:**
   - при росте $n$ сплайн обычно даёт меньшую RMSE;
   - осцилляции полинома у концов интервала выражены сильнее (проявление эффекта Рунге).

2. **Сходимость сплайна на $f(x)$:**
   - RMSE убывает при увеличении числа узлов;
   - наклон на log-log графике соответствует экспериментальной скорости сходимости.

3. **Равномерные узлы vs узлы Чебышёва:**
   - узлы Чебышёва обычно дают более устойчивую аппроксимацию на краях интервала;
   - сравнение наклонов и RMSE-таблицы показывает влияние выбора узлов на точность.
